# PicoCal - Pairwise-interaction transformer, ParT-style (notebook 15)

The research-grounded architecture. From our 2026 deep-research report:

> *Particle-Transformer-style attention adds a learned **pairwise-interaction matrix U** as an additive **pre-softmax** bias, `P-MHA = SoftMax(QKt/sqrt(d) + U)V`, which beats both Deep-Sets/PFN and GravNet-style ParticleNet on jet classification (arXiv:2202.03772).*

We fuse that mechanism with the recipe that already won min-bias in nb13 - energy-weighted **EFN sum pooling** + **residual/bias target** on top of the calibrated sum (CMS DeepSC, arXiv:2204.10277) + cluster-level global features.

**Why U helps under pileup:** plain self-attention only sees each cell's own features. The pairwise matrix U injects *cell-cell* physics - are two cells close (small dR) and both energetic? then they likely belong to the same photon shower; a far, low-energy cell is likely pileup. U lets attention reason about *relationships*, which is exactly what separates photon cells from background. A blind sum cannot; plain attention barely can.

**Ablation:** the flagship `PairT` vs the same core with `U=0` (which reduces exactly to nb13's EfnResidual winner). That isolates the contribution of the pairwise bias.

Goal: push min-bias `sigma_eff` toward 0.02. Honest floor caveat in the read-out cell - clean signal floored near 0.036 and pileup is harder, so 0.02 may sit below the achievable limit; the deliverable is the best research-grounded architecture + an honest read of where the floor is.

In [ ]:
import sys, copy, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, split, resolution, EPS

SEEDS = 8
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 150, "patience": 25, "pair_hidden": 32}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cache = repo / "data" / "cache" / "minbias_knn25.pkl"
if cache.exists():
    with open(cache, "rb") as f:
        D = pickle.load(f)
    print("loaded cache:", cache.name)
else:
    files = sorted((repo / "data" / "minimum_bias").glob("matched_*.root"))
    D = build(files, 3, 100.0, selector=lambda c: select_knn(c, 25))
    print("built from root (no cache)")
toks = D["tok_seed"]; y = D["y"]; Et = D["Etrue"]; region = D["region"]; agg = D["agg"]
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))
in_dim = toks[int(keep[0])].shape[1]
{"clusters": int(len(y)), "kept": int(len(keep)), "in_dim": int(in_dim), "device": DEVICE}

In [ ]:
G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
n_global = G.shape[1]
la, lb = np.polyfit(agg[ktr, 0], y[ktr], 1)
base_all = (la * agg[:, 0] + lb).astype(np.float32)

gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[ktr], y[ktr])
BDT = float(resolution(np.exp(gb.predict(agg[kte])), Et[kte])["sigma_eff"])
SUMcal = float(resolution(np.exp(la * agg[kte, 0] + lb), Et[kte])["sigma_eff"])

maxL = max(t.shape[0] for t in toks)
N = len(toks)
Xall = np.zeros((N, maxL, in_dim), np.float32)
Mall = np.zeros((N, maxL), np.bool_)
Wall = np.zeros((N, maxL), np.float32)
Rall = np.zeros((N, maxL, 3), np.float32)   # raw [rel_x, rel_y, log_energy] for pairwise features
for i, t in enumerate(toks):
    L = t.shape[0]; Xall[i, :L] = t; Mall[i, :L] = True
    Rall[i, :L, 0] = t[:, 3]; Rall[i, :L, 1] = t[:, 4]; Rall[i, :L, 2] = t[:, 0]
    e = np.expm1(np.clip(t[:, 0], 0, None)); Wall[i, :L] = e / (e.sum() + 1e-9)
cont = np.concatenate([toks[i][:, :7] for i in ktr], 0); tmean = cont.mean(0); tstd = cont.std(0) + EPS
Xall[:, :, :7] = (Xall[:, :, :7] - tmean) / tstd
Xall[~Mall] = 0.0
gmean = G[ktr].mean(0); gstd = G[ktr].std(0) + EPS
Gall = ((G - gmean) / gstd).astype(np.float32)

Xt = torch.from_numpy(Xall).to(DEVICE); Mt = torch.from_numpy(Mall).to(DEVICE)
Wt = torch.from_numpy(Wall).to(DEVICE); Gt = torch.from_numpy(Gall).to(DEVICE)
Rt = torch.from_numpy(Rall).to(DEVICE)
Bt = torch.from_numpy(base_all).unsqueeze(1).to(DEVICE)
Yt = torch.from_numpy(y.astype(np.float32)).unsqueeze(1).to(DEVICE)
{"BDT": round(BDT, 4), "sum_calib": round(SUMcal, 4), "n_train": int(len(ktr)),
 "n_test": int(len(kte)), "maxL": int(maxL)}

## The architecture
`pair_features(R)` turns per-cell `[rel_x, rel_y, log E]` into 5 physical **pair** features - `dx, dy, dR, logE_i+logE_j, min(logE_i,logE_j)` - the calorimeter-cell analogue of ParT's particle-pair `(dR, kT, mass)`. A small MLP maps those to one bias scalar **per head**, giving `U` of shape `(B, nhead, L, L)`, which is added to the attention scores **before** softmax in every block. Set `use_pair=False` and U is zero -> the model is exactly nb13's EfnResidual.

In [ ]:
def pair_features(R, m):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1)
    dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx * dx + dy * dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1)
    emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    return torch.stack([dx, dy, dR, esum, emin], -1)   # (B, L, L, 5)


class PairEmbed(nn.Module):
    def __init__(self, nhead, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(5, hidden), nn.GELU(),
                                 nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, nhead))
    def forward(self, pf):
        return self.net(pf).permute(0, 3, 1, 2).contiguous()   # (B, nhead, L, L)


class PMHA(nn.Module):
    def __init__(self, d, nhead, dropout):
        super().__init__()
        self.h = nhead; self.dh = d // nhead
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, U, key_valid):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5)
        if U is not None:
            scores = scores + U
        pad = (~key_valid).view(B, 1, 1, L)
        scores = scores.masked_fill(pad, -1e9)
        a = self.drop(scores.softmax(-1))
        out = (a @ v).transpose(1, 2).reshape(B, L, d)
        return self.o(out)


class Block(nn.Module):
    def __init__(self, d, nhead, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nhead, dropout)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, U, key_valid):
        x = x + self.drop(self.attn(self.n1(x), U, key_valid))
        x = x + self.drop(self.ff(self.n2(x)))
        return x


class PairT(nn.Module):
    def __init__(self, use_pair=True):
        super().__init__()
        d = cfg["d"]; self.use_pair = use_pair
        self.embed = nn.Linear(in_dim, d)
        self.pair = PairEmbed(cfg["nhead"], cfg["pair_hidden"]) if use_pair else None
        self.blocks = nn.ModuleList([Block(d, cfg["nhead"], cfg["dropout"]) for _ in range(cfg["layers"])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + n_global, d), nn.ReLU(),
                                  nn.Dropout(cfg["dropout"]), nn.Linear(d, 1))
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_features(R, m)) if self.use_pair else None
        h = self.embed(x)
        for blk in self.blocks:
            h = blk(h, U, m)
        p = self.norm((h * w.unsqueeze(-1)).sum(1))          # energy-weighted EFN pooling
        return base + self.head(torch.cat([p, g], 1))        # residual / bias target

In [ ]:
def train_eval(use_pair, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = PairT(use_pair=use_pair).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    def batches(idx, bs, shuffle):
        idx = np.asarray(idx)
        if shuffle:
            idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield Xt[b], Mt[b], Wt[b], Gt[b], Bt[b], Rt[b], Yt[b]

    def run(idx):
        out = []
        with torch.no_grad():
            for X, m, w, g, base, R, _ in batches(idx, 512, False):
                out.append(model(X, m, w, g, base, R).cpu().numpy().ravel())
        return np.concatenate(out)

    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, w, g, base, R, yb in batches(kva, 512, False):
                s += nn.functional.mse_loss(model(X, m, w, g, base, R), yb).item(); c += 1
        return s / max(c, 1)

    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, w, g, base, R, yb in batches(ktr, cfg["batch"], True):
            opt.zero_grad()
            nn.functional.mse_loss(model(X, m, w, g, base, R), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4:
            best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b)
    return float(resolution(pe, Et[kte])["sigma_eff"]), model, (float(a), float(b))

In [ ]:
VARIANTS = {"PairT (with pairwise U)": True, "EfnResidual (U=0 ablation)": False}
res15 = {}; models = {}; calib = {}; rows = []
t0 = time.time()
for name, up in VARIANTS.items():
    vals = []
    for s in range(SEEDS):
        sig, mdl, ab = train_eval(up, s)
        vals.append(sig)
        if s == 0:
            models[name] = mdl; calib[name] = ab
        print(f"  {name} seed {s}: {sig:.4f}", flush=True)
    mean, std = float(np.mean(vals)), float(np.std(vals))
    res15[name] = vals
    rows.append({"variant": name, "sigma_eff": round(mean, 4), "std": round(std, 4),
                 "BDT": round(BDT, 4), "beats_BDT": mean < BDT})
    print(f"{name}: {mean:.4f} +/- {std:.4f} {'BEATS' if mean < BDT else 'loses'} BDT {BDT:.4f}", flush=True)
print(f"elapsed {time.time()-t0:.0f}s")
summary15 = pd.DataFrame(rows).sort_values("sigma_eff").reset_index(drop=True)
summary15

In [ ]:
import plotly.graph_objects as go
d = summary15
colors = ["#1f77b4" if "PairT" in v else "#8c8c8c" for v in d["variant"]]
fig = go.Figure(go.Bar(x=d["variant"], y=d["sigma_eff"], error_y=dict(type="data", array=d["std"]),
                       marker_color=colors, text=[f"{v:.4f}" for v in d["sigma_eff"]], textposition="outside"))
fig.add_hline(y=BDT, line_dash="dash", line_color="crimson", annotation_text=f"fair BDT {BDT:.4f}")
fig.add_hline(y=0.02, line_dash="dot", line_color="green", annotation_text="goal 0.02")
fig.update_layout(template="plotly_white", height=460, yaxis_title="sigma_eff (min-bias)",
                  title="Pairwise-interaction bias vs the EfnResidual ablation (min-bias)")
fig.show()

In [ ]:
def sigma_vs_energy(pred_e, true_e, n_bins=8):
    edges = np.quantile(true_e, np.linspace(0, 1, n_bins + 1))
    cx, sy = [], []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-6 if i == n_bins - 1 else 0.0)
        mk = (true_e >= edges[i]) & (true_e < hi)
        if mk.sum() >= 20:
            cx.append(float(np.median(true_e[mk]))); sy.append(float(resolution(pred_e[mk], true_e[mk])["sigma_eff"]))
    return np.array(cx), np.array(sy)

def predict_e(name):
    model = models[name]; a, b = calib[name]; rng = np.random.default_rng(0)
    idx = np.asarray(kte); out = []
    with torch.no_grad():
        for j in range(0, len(idx), 512):
            bb = torch.from_numpy(idx[j:j + 512]).to(DEVICE)
            out.append(model(Xt[bb], Mt[bb], Wt[bb], Gt[bb], Bt[bb], Rt[bb]).cpu().numpy().ravel())
    return np.exp(a * np.concatenate(out) + b)

best = summary15.iloc[0]["variant"]
curves = {best: predict_e(best), "BDT": np.exp(gb.predict(agg[kte])), "sum-calib": np.exp(la * agg[kte, 0] + lb)}
cols = {best: "#1f77b4", "BDT": "#8c8c8c", "sum-calib": "#d62728"}
fig = go.Figure()
for nm, pe in curves.items():
    cx, sy = sigma_vs_energy(pe, Et[kte])
    fig.add_trace(go.Scatter(x=cx, y=sy, mode="lines+markers", name=nm, line=dict(color=cols[nm])))
fig.update_layout(template="plotly_white", height=430, xaxis_title="E_true [GeV] (bin median)",
                  yaxis_title="sigma_eff", title="Resolution vs energy (adaptive bins) - best model vs baselines")
fig.show()

## Read-out
- **Did the pairwise bias U help?** `PairT` vs the `U=0` ablation isolates it. A real gain means cell-cell relationships carry information the per-cell tokens + sum do not - the ParT thesis, now on a calorimeter under pileup.
- **vs the goal 0.02:** report the number honestly. If we land above 0.02, state the achieved value and that 0.02 likely sits below the pileup floor (clean-signal floor ~0.036; adding background cannot improve intrinsic resolution). The win that matters is the margin over BDT and the sum, and whether U widens it further.
- Next levers if more is wanted: more blocks / heads, add ParT class-token pooling, or a GravNet graph as an independent cross-check.